In [ ]:
from pathlib import Path

import prism

import prism


In [2]:
def get_buildings_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [3]:
if not prep_fp.is_file():
    prep_data = preprocess(base_dir)
    export_to_netcdf(prep_data, prep_fp)
else:
    prep_data = import_from_netcdf(prep_fp)

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
main_model_factory = ModelFactory(
    new_prep_data, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    ).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)

In [ ]:
list(main_model_factory.default)

In [ ]:
scenario_list = {"base":("SSP2",["base"])}

In [ ]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dirs = {
        scenario: scenario_base_path / scenario for scenario in circular_scen
    }

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs) 




    factory = ModelFactory(
    [bld_sector], complete_timeline
    ).add(GenericStocks, ["buildings"]
    ).add(MaterialIntensities, "buildings",
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    print(f"Finished {scen_id}")

In [ ]:
bld_sector.prep_data.get("stocks").Type.values

In [ ]:
import matplotlib.pyplot as plt


bld_sector.prep_data.get("stocks").sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

bld_sector.prep_data.get("stocks").sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')
plt.legend()

In [ ]:
model.buildings.keys()

In [ ]:
model.buildings.get("inflow").to_array().sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow").to_array().sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array().sel(material = 'Steel', Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow_materials").to_array().sel(material = 'Steel', Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.save_pkl("test.pkl")
model2 = ModelFactory.load_pkl("test.pkl")
model2.buildings["inflow_materials"]